# Farmer example

Reference: Birge and Louveaux, "Introduction to Stochastic Programming", Chapter 1

The implementation directly follows the documentation of StochasticPrograms.jl: https://martinbiel.github.io/StochasticPrograms.jl/stable/manual/examples/

In [1]:
# the environment of this folder, which uses the fixed fork of StochasticPrograms
import Pkg
Pkg.activate(".")

  Activating project at `~/Git/Optimization/SP_Introduction/code/StochasticPrograms`


In [2]:
using StochasticPrograms
using HiGHS

[ Info: Precompiling StochasticPrograms [8b8459f2-c380-502b-8633-9aed2d6c2b35] (cache misses: wrong dep version loaded (8), wrong source (6), mismatched flags (4))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
[ Info: Precompiling HiGHS [87dc4568-4c63-4d18-b0c0-bb2238e4078b] (cache misses: wrong dep version loaded (8), incompatible header (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


We will add some benchmark tools in order to better evaluate the efficiency of each approach.

In [3]:
using BenchmarkTools

[ Info: Precompiling BenchmarkTools [6e4b80f9-dd63-53aa-95a3-0cdb28fa8baf] (cache misses: wrong dep version loaded (12), incompatible header (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
[ Info: Precompiling MathOptInterfaceBenchmarkToolsExt [d01049fa-8689-518c-8a9f-1b588bc0d871] (cache misses: wrong dep version loaded (6))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


Using StochasticPrograms.jl's syntax, we describe the two-stage linear program.

In [4]:
Crops = [:wheat, :corn, :beets]
@stochastic_model farmer_model begin
    @stage 1 begin
        @parameters begin
            Crops = Crops
            Cost = Dict(:wheat=>150, :corn=>230, :beets=>260)
            Budget = 500
        end
        @decision(farmer_model, x[c in Crops] >= 0)
        @objective(farmer_model, Min, sum(Cost[c]*x[c] for c in Crops))
        @constraint(farmer_model, sum(x[c] for c in Crops) <= Budget)
    end
    @stage 2 begin
        @parameters begin
            Crops = Crops
            Required = Dict(:wheat=>200, :corn=>240, :beets=>0)
            PurchasePrice = Dict(:wheat=>238, :corn=>210)
            SellPrice = Dict(:wheat=>170, :corn=>150, :beets=>36, :extra_beets=>10)
        end
        @uncertain ξ[c in Crops]
        @recourse(farmer_model, y[p in setdiff(Crops, [:beets])] >= 0)
        @recourse(farmer_model, w[s in Crops ∪ [:extra_beets]] >= 0)
        @objective(farmer_model, Min, sum(PurchasePrice[p] * y[p] for p in setdiff(Crops, [:beets]))
                   - sum(SellPrice[s] * w[s] for s in Crops ∪ [:extra_beets]))
        @constraint(farmer_model, minimum_requirement[p in setdiff(Crops, [:beets])],
            ξ[p] * x[p] + y[p] - w[p] >= Required[p])
        @constraint(farmer_model, minimum_requirement_beets,
            ξ[:beets] * x[:beets] - w[:beets] - w[:extra_beets] >= Required[:beets])
        @constraint(farmer_model, beets_quota, w[:beets] <= 6000)
    end
end

Two-Stage Stochastic Model

minimize f₀(x) + 𝔼[f(x,ξ)]
  x∈𝒳

where

f(x,ξ) = min  f(y; x, ξ)
              y ∈ 𝒴 (x, ξ)


The package allows to formulate the program as a classical two-stage stochastic program. We now have to provide the scenarios.

In [5]:
ξ₁ = @scenario ξ[c in Crops] = [3.0, 3.6, 24.0] probability = 1/3
ξ₂ = @scenario ξ[c in Crops] = [2.5, 3.0, 20.0] probability = 1/3
ξ₃ = @scenario ξ[c in Crops] = [2.0, 2.4, 16.0] probability = 1/3

Scenario with probability 0.3333333333333333 and underlying data:

1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, [:wheat, :corn, :beets]
And data, a 3-element Vector{Float64}:
  2.0
  2.4
 16.0

## Deterministic equivalent

We finally pass the complete model to our solver. By default, the deterministic equivalent problem is built.

In [6]:
farmer = instantiate(farmer_model, [ξ₁,ξ₂,ξ₃], optimizer = HiGHS.Optimizer)

Stochastic program with:
 * 3 decision variables
 * 6 recourse variables
 * 3 scenarios of type Scenario
Structure: Deterministic equivalent
Solver name: HiGHS

We can check the model formulation.

In [7]:
println(farmer)

Deterministic equivalent problem
Min 150 x[wheat] + 230 x[corn] + 260 x[beets] + 79.33333333333333 y₁[wheat] + 70 y₁[corn] - 56.666666666666664 w₁[wheat] - 50 w₁[corn] - 12 w₁[beets] - 3.333333333333333 w₁[extra_beets] + 79.33333333333333 y₂[wheat] + 70 y₂[corn] - 56.666666666666664 w₂[wheat] - 50 w₂[corn] - 12 w₂[beets] - 3.333333333333333 w₂[extra_beets] + 79.33333333333333 y₃[wheat] + 70 y₃[corn] - 56.666666666666664 w₃[wheat] - 50 w₃[corn] - 12 w₃[beets] - 3.333333333333333 w₃[extra_beets]
Subject to
 x[wheat] ∈ Decisions
 x[corn] ∈ Decisions
 x[beets] ∈ Decisions
 y₁[wheat] ∈ RecourseDecisions
 y₁[corn] ∈ RecourseDecisions
 w₁[wheat] ∈ RecourseDecisions
 w₁[corn] ∈ RecourseDecisions
 w₁[beets] ∈ RecourseDecisions
 w₁[extra_beets] ∈ RecourseDecisions
 y₂[wheat] ∈ RecourseDecisions
 y₂[corn] ∈ RecourseDecisions
 w₂[wheat] ∈ RecourseDecisions
 w₂[corn] ∈ RecourseDecisions
 w₂[beets] ∈ RecourseDecisions
 w₂[extra_beets] ∈ RecourseDecisions
 y₃[wheat] ∈ RecourseDecisions
 y₃[corn] ∈ Re

We can now run the optimization solver. We also print the first-stage decision as well as the optimal value.

In [8]:
optimize!(farmer)
x = optimal_decision(farmer)
x = farmer[1,:x]
println("Wheat: $(value(x[:wheat]))")
println("Corn: $(value(x[:corn]))")
println("Beets: $(value(x[:beets]))")
println("Cost: $(objective_value(farmer))")

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
LP has 13 rows; 21 cols; 33 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 2e+01]
  Cost    [3e+00, 3e+02]
  Bound   [0e+00, 0e+00]
  RHS     [2e+02, 6e+03]
Solving LP with useful basis so presolve not used
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0    -3.4858331896e+02 Ph1: 13(15.325); Du: 9(348.583) 0.0s
         13    -1.0839000000e+05 Pr: 0(0) 0.0s

Model status        : Optimal
Simplex   iterations: 13
Objective value     : -1.0839000000e+05
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.01
Wheat: 170.00000000000006
Corn: 80.0
Beets: 249.99999999999997
Cost: -108389.99999999996


The second-stage solutions for the first scenario are

In [9]:
s = 3  # scenario number
y = farmer[2,:y]
w = farmer[2,:w]
println("Purchased wheat: $(value(y[:wheat], s))")
println("Purchased corn: $(value(y[:corn], s))")
println("Sold wheat: $(value(w[:wheat], s))")
println("Sold corn: $(value(w[:corn], s))")
println("Sold beets: $(value(w[:beets], s))")
println("Sold extra beets: $(value(w[:extra_beets], s))")
println("Profit: $(objective_value(farmer, s))")

Purchased wheat: 0.0
Purchased corn: 48.0
Sold wheat: 140.00000000000009
Sold corn: 0.0
Sold beets: 3999.999999999999
Sold extra beets: 0.0
Profit: -157720.0


`StochasticPrograms` also allows to compute the expected value of perfect information...

In [10]:
println("EVPI: $(EVPI(farmer))")

LP has 13 rows; 21 cols; 33 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 2e+01]
  Cost    [3e+00, 3e+02]
  Bound   [0e+00, 0e+00]
  RHS     [2e+02, 6e+03]
Solving LP with useful basis so presolve not used
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0    -3.4858331896e+02 Ph1: 13(15.325); Du: 9(348.583) 0.0s
         13    -1.0839000000e+05 Pr: 0(0) 0.0s

Model status        : Optimal
Simplex   iterations: 13
Objective value     : -1.0839000000e+05
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00
Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
LP has 5 rows; 9 cols; 13 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 2e+01]
  Cost    [1e+01, 3e+02]
  Bound   [0e+00, 0e+00]
  RHS     [2e+02, 6e+03]
Presolving model
4 rows, 9 cols, 12 nonzeros 0s
3 rows, 8 co

... and the value of the stochastic solution.

In [11]:
println("VSS: $(VSS(farmer))")

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
LP has 5 rows; 9 cols; 13 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 2e+01]
  Cost    [1e+01, 3e+02]
  Bound   [0e+00, 0e+00]
  RHS     [2e+02, 6e+03]
Presolving model
4 rows, 9 cols, 12 nonzeros 0s
3 rows, 8 cols, 10 nonzeros 0s
Dependent equations search running on 2 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
3 rows, 7 cols, 9 nonzeros 0s
Presolve reductions: rows 3(-2); columns 7(-2); nonzeros 9(-4) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0    -6.3999740210e+02 Ph1: 2(2); Du: 2(639.997) 0.0s
          4    -1.1860000000e+05 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Mode

## L-shaped decomposition

Instead of the deterministic equivalent, we can use the L-shaped decomposition method.

### Multi-cuts

Let gather the scenario in one vector.

In [12]:
ξ = [ξ₁, ξ₂, ξ₃]

3-element Vector{Scenario{JuMP.Containers.DenseAxisArray{Float64, 1, Tuple{Vector{Symbol}}, Tuple{JuMP.Containers._AxisLookup{Dict{Symbol, Int64}}}}}}:
 Scenario with probability 0.3333333333333333 and underlying data:

1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, [:wheat, :corn, :beets]
And data, a 3-element Vector{Float64}:
  3.0
  3.6
 24.0
 Scenario with probability 0.3333333333333333 and underlying data:

1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, [:wheat, :corn, :beets]
And data, a 3-element Vector{Float64}:
  2.5
  3.0
 20.0
 Scenario with probability 0.3333333333333333 and underlying data:

1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, [:wheat, :corn, :beets]
And data, a 3-element Vector{Float64}:
  2.0
  2.4
 16.0

We instantitate the model with the L-shaped method by specifying it as the chosen optimizer. By default, the multi-cut approach is selected.

In [13]:
farmer_lshaped = instantiate(farmer_model, ξ, optimizer = LShaped.Optimizer)

Stochastic program with:
 * 3 decision variables
 * 6 recourse variables
 * 3 scenarios of type Scenario
Structure: Stage-decomposition
Solver name: L-shaped with disaggregate cuts

The model is now explicitly a two-stage model.

In [14]:
print(farmer_lshaped)

First-stage 
Min 150 x[wheat] + 230 x[corn] + 260 x[beets]
Subject to
 x[wheat] ∈ Decisions
 x[corn] ∈ Decisions
 x[beets] ∈ Decisions
 x[wheat] ≥ 0
 x[corn] ≥ 0
 x[beets] ≥ 0
 x[wheat] + x[corn] + x[beets] ≤ 500

Second-stage 
Subproblem 1 (p = 0.33):
Min 238 y[wheat] + 210 y[corn] - 170 w[wheat] - 150 w[corn] - 36 w[beets] - 10 w[extra_beets]
Subject to
 x[wheat] ∈ Known(value = 0.0)
 x[corn] ∈ Known(value = 0.0)
 x[beets] ∈ Known(value = 0.0)
 y[wheat] ∈ RecourseDecisions
 y[corn] ∈ RecourseDecisions
 w[wheat] ∈ RecourseDecisions
 w[corn] ∈ RecourseDecisions
 w[beets] ∈ RecourseDecisions
 w[extra_beets] ∈ RecourseDecisions
 y[wheat] ≥ 0
 y[corn] ≥ 0
 w[wheat] ≥ 0
 w[corn] ≥ 0
 w[beets] ≥ 0
 w[extra_beets] ≥ 0
 minimum_requirement[wheat] : 3 x[wheat] + y[wheat] - w[wheat] ≥ 200
 minimum_requirement[corn] : 3.6 x[corn] + y[corn] - w[corn] ≥ 240
 minimum_requirement_beets : 24 x[beets] - w[beets] - w[extra_beets] ≥ 0
 beets_quota : w[beets] ≤ 6000

Subproblem 2 (p = 0.33):
Min 238 y[wh

We have to specify the solver for each stage. This allows to choose more adapted algorithms

In [15]:
using GLPK

[ Info: Precompiling GLPK [60bf3e95-4087-53dc-ae20-288a0d20c6a6] (cache misses: wrong dep version loaded (10), mismatched flags (4))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [16]:
set_optimizer_attribute(farmer_lshaped, MasterOptimizer(), GLPK.Optimizer)
set_optimizer_attribute(farmer_lshaped, SubProblemOptimizer(), HiGHS.Optimizer)

We are now in position to solve the program. `StochasticPrograms` informs us on the number of iterations and generated cuts.

In [17]:
optimize!(farmer_lshaped)

┌ Warning: ProgressMeter by default refresh meters with additional information in IJulia via `IJulia.clear_output`, which clears all outputs in the cell. 
│  - To prevent this behaviour, do `ProgressMeter.ijulia_behavior(:append)`. 
│  - To disable this warning message, do `ProgressMeter.ijulia_behavior(:clear)`.
└ @ ProgressMeter ~/.julia/packages/ProgressMeter/N660J/src/ProgressMeter.jl:607
L-Shaped Gap  Time: 0:00:02 (6 iterations)
        Objective: -108390.0
              Gap: 2.685102911406373e-16
   Number of cuts: 14
       Iterations: 6


Let measure the performances.

In [18]:
@benchmark optimize!(farmer_lshaped)

┌ Warning: ProgressMeter by default refresh meters with additional information in IJulia via `IJulia.clear_output`, which clears all outputs in the cell. 
│  - To prevent this behaviour, do `ProgressMeter.ijulia_behavior(:append)`. 
│  - To disable this warning message, do `ProgressMeter.ijulia_behavior(:clear)`.
└ @ ProgressMeter ~/.julia/packages/ProgressMeter/N660J/src/ProgressMeter.jl:607
L-Shaped Gap  Time: 0:00:00 (6 iterations)
        Objective: -108390.0
              Gap: 2.685102911406373e-16
   Number of cuts: 14
       Iterations: 6


BenchmarkTools.Trial: 188 samples with 1 evaluation per sample.
 Range (min … max):  23.239 ms … 140.205 ms  ┊ GC (min … max): 0.00% … 47.15%
 Time  (median):     25.654 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   26.601 ms ±   8.484 ms  ┊ GC (mean ± σ):  1.32% ±  3.44%

            ▅▅█▂ ▂▇▂▂                                           
  ▃▁▁▁▁▃▃▄▆▆█████████▇▆▅▅▃▃▃▃▃▁▃▃▃▃▁▁▁▁▁▁▃▄▁▁▁▁▃▃▁▁▃▁▁▁▃▁▁▃▁▃▃ ▃
  23.2 ms         Histogram: frequency by time         32.6 ms <

 Memory estimate: 1.51 MiB, allocs estimate: 42793.

We want to compare it with the simple cut version, so we ask the program to aggregate the cuts.

In [19]:
set_optimizer_attribute(farmer_lshaped, Aggregator(), Aggregate())

If we solve the problem, we now see that we need more iterations, while a few less cuts are generated. As expected, one cut is now produced at each iteration, except the last one, that established the convergence.

In [20]:
optimize!(farmer_lshaped)

┌ Warning: ProgressMeter by default refresh meters with additional information in IJulia via `IJulia.clear_output`, which clears all outputs in the cell. 
│  - To prevent this behaviour, do `ProgressMeter.ijulia_behavior(:append)`. 
│  - To disable this warning message, do `ProgressMeter.ijulia_behavior(:clear)`.
└ @ ProgressMeter ~/.julia/packages/ProgressMeter/N660J/src/ProgressMeter.jl:607
L-Shaped Gap  Time: 0:00:00 (11 iterations)
        Objective: -108389.99999999994
              Gap: 2.6851029114063745e-16
   Number of cuts: 10
       Iterations: 11


The benchmark also exhibits that the single cut technique is significantly slower.

In [21]:
@benchmark optimize!(farmer_lshaped)

┌ Warning: ProgressMeter by default refresh meters with additional information in IJulia via `IJulia.clear_output`, which clears all outputs in the cell. 
│  - To prevent this behaviour, do `ProgressMeter.ijulia_behavior(:append)`. 
│  - To disable this warning message, do `ProgressMeter.ijulia_behavior(:clear)`.
└ @ ProgressMeter ~/.julia/packages/ProgressMeter/N660J/src/ProgressMeter.jl:607
L-Shaped Gap  Time: 0:00:00 (11 iterations)
        Objective: -108389.99999999994
              Gap: 2.6851029114063745e-16
   Number of cuts: 10
       Iterations: 11


BenchmarkTools.Trial: 195 samples with 1 evaluation per sample.
 Range (min … max):  23.626 ms … 129.495 ms  ┊ GC (min … max): 0.00% … 48.01%
 Time  (median):     24.915 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   25.745 ms ±   7.546 ms  ┊ GC (mean ± σ):  1.24% ±  3.44%

       ▄▂▆▇█▂▄█▃▄▆   ▂▃ ▁    ▁                                  
  ▃▁▃▄█████████████▆▆██▄██▆▃▆█▆▁▆▃▃▃▃▃▆▁▄▄▁▁▁▃▁▁▁▁▃▁▃▃▃▄▁▁▄▁▁▃ ▃
  23.6 ms         Histogram: frequency by time           29 ms <

 Memory estimate: 1.64 MiB, allocs estimate: 44864.